# SVD (Singular Value Decomposition)

Es un algoritmo de filtrado colaborativo basado en factorización matricial que representa usuarios e ítems mediante factores latentes aprendidos a partir de sus valoraciones, permitiendo predecir las preferencias de los usuarios sobre ítems que aún no ha valorado.

In [1]:
import pandas as pd
import numpy as np

In [2]:
ratings_df = pd.read_csv('ratings_limpios.csv')
resumen_usuario = pd.read_csv('resumen_usuario.csv')

In [36]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

reader = Reader(rating_scale=(1,10))
surprise_df = ratings_df.copy()
surprise_df.columns = ["uid", "iid", "rating"]

data = Dataset.load_from_df(surprise_df, reader)

# 0.1, 0.2, 0.3
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

algo = SVD()
algo.fit(trainset)
predictions = algo.test(testset)

In [37]:
# Transformación a dataframe
preds_df = pd.DataFrame([
    {
        'User-ID': pred.uid,
        'ISBN': pred.iid,
        'r_ui': pred.r_ui,
        'est': pred.est
    }
    for pred in predictions
])

In [38]:
from metrics import evaluar_metricas_usuario

K = 10
metricas_usuarios = (
    preds_df
    .groupby('User-ID')
    .apply(evaluar_metricas_usuario, k=K)
    .reset_index()
)


df_evaluacion = pd.merge(
    metricas_usuarios,
    resumen_usuario,
    on='User-ID',
    how='inner',
    validate='one_to_one'
)

In [39]:
# A. Por Grupo Etario (Demográfico)
print(f"=== Métricas por Grupo Etario (K={K}) ===")
print(df_evaluacion.groupby('Grupo_Etario')[['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']].mean())

# B. Por Historial de Interacciones (Comportamiento)
print(f"\n=== Métricas por Historial de Interacciones (0: Corto, 1: Largo) (K={K}) ===")
print(df_evaluacion.groupby('Grupo_Historial')[['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']].mean())

# C. Por Grado de Exigencia (Comportamiento)
print(f"\n=== Métricas por Grado de Exigencia (0: Exigente, 1: Normal, 2: Generoso) (K={K}) ===")
print(df_evaluacion.groupby('Grupo_Exigencia')[['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']].mean())

=== Métricas por Grupo Etario (K=10) ===
                   MAE      CG@10     DCG@10   NDCG@10
Grupo_Etario                                          
Adultos       1.345677  19.924433  13.257177  0.980844
Jóvenes       1.375250  17.355240  12.266225  0.985067
Mayores       1.333692  14.925430  11.248892  0.988423

=== Métricas por Historial de Interacciones (0: Corto, 1: Largo) (K=10) ===
                      MAE      CG@10     DCG@10   NDCG@10
Grupo_Historial                                          
0                1.405584   7.817710   7.687137  0.999267
1                1.346255  21.598986  14.110846  0.977379

=== Métricas por Grado de Exigencia (0: Exigente, 1: Normal, 2: Generoso) (K=10) ===
                      MAE      CG@10     DCG@10   NDCG@10
Grupo_Exigencia                                          
0                2.492117   8.709990   6.838612  0.985768
1                1.144769  18.587971  12.749364  0.982423
2                1.825232  17.756060  13.479571  0.995713

Obserbaciones:

Grupo Etario:

Representa el agrupamiento más homogéneo. Si bien se observan ciertas diferencias entre los tres grupos, estos son relativamente pequeñas. Estas diferencias podrían estar relacionadas con la cantidad de valoraciones disponibles para cada grupo, ya que un número desigual de observaciones puede afectar la estabilidad de la métricas obtenidas.


Grupo separado por historial:

El sistema presenta una diferencia marcada en su desempeño según la longitud del historial de los usuarios, favoreciendo considerablemente aquellos con un historial largo. El valor de NDGC aproximado a 1 indica que, en los casos evaluados, los ítems relevantes fueron posicionados coreectamente. Sin embargo, este resultado debe tomarse con precaución, ya que muchos usuarios poseen muy pocos ítems en el conjunto de prueba, incluso en algunos casos solo uno, lo que puede hacer que alcanzar un NDCG alto sea sencillo.


Grado de exigencia:

El sistema presenta dificultades para estimar con precisión las calificaciones de los usuarios exigentes (Grupo 0), sobreestimando sus valoraciones y empeorando el error (MAE). En contraste, los usuarios normales y generosos (Grupo 1 y 2) resultan altamente beneficiados, obteniendo los mejores valores en todas las métricas debido a la menor varianza de sus evaluaciones. Destacando que los usuarios normales tienen mejor MAE.

### Hipótesis: los usuarios con un historial de interacciones largo se ven más beneficiados por el sistema de recomendación que aquellos con un historial corto.

In [17]:
from scipy.stats import shapiro

g_corto = df_evaluacion[df_evaluacion['Grupo_Historial'] == 0]
g_largo = df_evaluacion[df_evaluacion['Grupo_Historial'] == 1]

metricas = ['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

for metrica in metricas:
    # Extracción de valores limpios
    v_corto = g_corto[metrica].dropna()
    v_largo = g_largo[metrica].dropna()

    print(f"\n {metrica}")

    stat, p = shapiro(v_corto)
    print(f"Test de Shapiro-Wilk — Historial corto — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    stat, p = shapiro(v_largo)
    print(f"Test de Shapiro-Wilk — Historial largo — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")


 MAE
Test de Shapiro-Wilk — Historial corto — MAE: estadístico=0.889, p-valor=0.000
Test de Shapiro-Wilk — Historial largo — MAE: estadístico=0.918, p-valor=0.000

 CG@10
Test de Shapiro-Wilk — Historial corto — CG@10: estadístico=0.574, p-valor=0.000
Test de Shapiro-Wilk — Historial largo — CG@10: estadístico=0.731, p-valor=0.000

 DCG@10
Test de Shapiro-Wilk — Historial corto — DCG@10: estadístico=0.854, p-valor=0.000
Test de Shapiro-Wilk — Historial largo — DCG@10: estadístico=0.840, p-valor=0.000

 NDCG@10
Test de Shapiro-Wilk — Historial corto — NDCG@10: estadístico=0.058, p-valor=0.000
Test de Shapiro-Wilk — Historial largo — NDCG@10: estadístico=0.586, p-valor=0.000


c:\Users\esper\Desktop\tp-recomendacion\venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 8278.
  res = hypotest_fun_out(*samples, **kwds)
c:\Users\esper\Desktop\tp-recomendacion\venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 18745.
  res = hypotest_fun_out(*samples, **kwds)


Ninguna de las métricas evaluadas mediante el test de Shapiro-Wilk presentó un p-valor mayor a 0,05, por lo que se rechaza la hipótesis de normalidad para todas las métricas.

In [10]:
import scipy.stats as stats

for metrica in metricas:
    # Extracción de valores limpios
    v_corto = g_corto[metrica].dropna()
    v_largo = g_largo[metrica].dropna()

    stat, p = stats.levene(v_largo,v_corto)
    print(f"Test de Levene para {metrica}: Estadístico={stat:.3f}, p-valor={p:.3f}")

Test de Levene para MAE: Estadístico=711.117, p-valor=0.000
Test de Levene para CG@10: Estadístico=3324.366, p-valor=0.000
Test de Levene para DCG@10: Estadístico=3640.165, p-valor=0.000
Test de Levene para NDCG@10: Estadístico=1972.168, p-valor=0.000


Mediante el test de Levene se determinó que las métricas de los grupos separados según la longitud del historial no presentan homocedasticidad.

Dado que no se cumplen los supuestos de normalidad ni de homocedasticidad, se recurre al test no paramétrico de Kruskal-Wallis.

In [20]:
alpha = 0.05

for metrica in metricas:
    # Extracción de valores limpios
    v_corto = g_corto[metrica].dropna()
    v_largo = g_largo[metrica].dropna()
    
    #Test de shapiro-wilk
    stat, p = stats.kruskal(v_largo, v_corto)
    print(f"\nTest de Kruskal-Wallis - {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    if p > alpha:
        print("No hay suficiente evidencia para rechazar la hipótesis nula.")
        print(f"No hay una diferencia significativa en {metrica} entre historiales largos y cortos.")
    else:
        print("Se rechaza la hipótesis nula.")
        print(f"Existe una diferencia significativa en {metrica} entre historiales largos y cortos.")
        


Test de Kruskal-Wallis - MAE: estadístico=8.804, p-valor=0.003
Se rechaza la hipótesis nula.
Existe una diferencia significativa en MAE entre historiales largos y cortos.

Test de Kruskal-Wallis - CG@10: estadístico=4284.478, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en CG@10 entre historiales largos y cortos.

Test de Kruskal-Wallis - DCG@10: estadístico=3984.848, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en DCG@10 entre historiales largos y cortos.

Test de Kruskal-Wallis - NDCG@10: estadístico=3447.497, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en NDCG@10 entre historiales largos y cortos.


La prueba de Kruskal-Wallis confirma diferencias estadísticamente significativas en todas las métricas evaluadas entre usuarios de historial corto y largo.

Los datos demuestran que el sistema de recomendación beneficia a los usuarios con un historial de interacciones largo, no solo logra reducir el error de predicción, sino que incrementa la cantidad y relevancia del contenido recomendado.

### Hipótesis: El modelo presenta un rendimiento similar en los distintos grupos etarios

In [16]:
from scipy.stats import shapiro

g_joven = df_evaluacion[df_evaluacion['Grupo_Etario'] == "Jóvenes"]
g_adulto = df_evaluacion[df_evaluacion['Grupo_Etario'] == "Adultos"]
g_mayores = df_evaluacion[df_evaluacion['Grupo_Etario'] == "Mayores"]

metricas = ['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

for metrica in metricas:
    
    v_joven = g_joven[metrica].dropna()
    v_adulto = g_adulto[metrica].dropna()
    v_mayores = g_mayores[metrica].dropna()

    print(f"\n {metrica}")

    stat, p = shapiro(v_joven)
    print(f"Test de Shapiro-Wilk — jóvenes — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    stat, p = shapiro(v_adulto)
    print(f"Test de Shapiro-Wilk — adultos — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    stat, p = shapiro(v_mayores)
    print(f"Test de Shapiro-Wilk — mayores — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")
        


 MAE
Test de Shapiro-Wilk — jóvenes — MAE: estadístico=0.910, p-valor=0.000
Test de Shapiro-Wilk — adultos — MAE: estadístico=0.908, p-valor=0.000
Test de Shapiro-Wilk — mayores — MAE: estadístico=0.932, p-valor=0.000

 CG@10
Test de Shapiro-Wilk — jóvenes — CG@10: estadístico=0.647, p-valor=0.000
Test de Shapiro-Wilk — adultos — CG@10: estadístico=0.684, p-valor=0.000
Test de Shapiro-Wilk — mayores — CG@10: estadístico=0.597, p-valor=0.000

 DCG@10
Test de Shapiro-Wilk — jóvenes — DCG@10: estadístico=0.779, p-valor=0.000
Test de Shapiro-Wilk — adultos — DCG@10: estadístico=0.796, p-valor=0.000
Test de Shapiro-Wilk — mayores — DCG@10: estadístico=0.753, p-valor=0.000

 NDCG@10
Test de Shapiro-Wilk — jóvenes — NDCG@10: estadístico=0.472, p-valor=0.000
Test de Shapiro-Wilk — adultos — NDCG@10: estadístico=0.534, p-valor=0.000
Test de Shapiro-Wilk — mayores — NDCG@10: estadístico=0.437, p-valor=0.000


c:\Users\esper\Desktop\tp-recomendacion\venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 13220.
  res = hypotest_fun_out(*samples, **kwds)


Ninguna de las métricas evaluadas mediante el test de Shapiro-Wilk presentó un p-valor mayor a 0,05, por lo que se rechaza la hipótesis de normalidad para todas las métricas.

In [18]:
import scipy.stats as stats

for metrica in metricas:
    
    v_joven = g_joven[metrica].dropna()
    v_adulto = g_adulto[metrica].dropna()
    v_mayores = g_mayores[metrica].dropna()
    
    stat, p = stats.levene(v_joven, v_adulto, v_mayores, center='median')
    print(f"Test de Levene para {metrica}: Estadístico={stat:.3f}, p-valor={p:.3f}")

Test de Levene para MAE: Estadístico=4.579, p-valor=0.010
Test de Levene para CG@10: Estadístico=34.622, p-valor=0.000
Test de Levene para DCG@10: Estadístico=35.244, p-valor=0.000
Test de Levene para NDCG@10: Estadístico=15.034, p-valor=0.000


Mediante el test de Levene se determinó que las métricas de los grupos separados según la longitud del historial no presentan homocedasticidad.

Dado que no se cumplen los supuestos de normalidad ni de homocedasticidad, se recurre al test no paramétrico de Kruskal-Wallis

In [21]:
alpha = 0.05

for metrica in metricas:
    # Extracción de valores limpios
    v_joven = g_joven[metrica].dropna()
    v_adulto = g_adulto[metrica].dropna()
    v_mayores = g_mayores[metrica].dropna()
    
    #Test de shapiro-wilk
    stat, p = stats.kruskal(v_joven, v_adulto, v_mayores)
    print(f"\nTest de Kruskal-Wallis — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    if p > alpha:
        print("No hay suficiente evidencia para rechazar la hipótesis nula.")
        print(f"No hay una diferencia significativa en {metrica} entre los grupos etarios.")
    else:
        print("Se rechaza la hipótesis nula.")
        print(f"Existe una diferencia significativa en {metrica} entre los grupos etarios.")


Test de Kruskal-Wallis — MAE: estadístico=1.489, p-valor=0.475
No hay suficiente evidencia para rechazar la hipótesis nula.
No hay una diferencia significativa en MAE entre los grupos etarios.

Test de Kruskal-Wallis — CG@10: estadístico=28.951, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en CG@10 entre los grupos etarios.

Test de Kruskal-Wallis — DCG@10: estadístico=24.195, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en DCG@10 entre los grupos etarios.

Test de Kruskal-Wallis — NDCG@10: estadístico=50.766, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en NDCG@10 entre los grupos etarios.


La hipótesis nula se mantiene únicamente para MAE, indicando homogeneidad en el error de predicción. Sin embargo, se rechaza para CG, DCG y NDCG, lo que demuestra la presencia de diferencias estadísticamente significativas enrte los grupos etarios en cuanto a la calidad y volumen de recomendación.

El modelo no presenta un rendimiento idéntico entre grupos etarios. Existe un sesgo a favor de los usuarios Adultos y Jóvenes en términos de la cantidad de contenido relevante recuperado. Para los usuarios Mayores el modelo logra ordenar mejor sus recomendaciones con un NDCG superior.

### Hipótesis: El modelo SVD presenta un rendimiento diferente según el grado de exigencia del usuario.

In [23]:
from scipy.stats import shapiro

#grupo de exigencia: 0 Exigente, 1 normal, 2 generoso
g_exigente = df_evaluacion[df_evaluacion['Grupo_Exigencia'] == 0]
g_normal = df_evaluacion[df_evaluacion['Grupo_Exigencia'] == 1]
g_generoso = df_evaluacion[df_evaluacion['Grupo_Exigencia'] == 2]

metricas = ['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

for metrica in metricas:
    
    v_exigente = g_exigente[metrica].dropna()
    v_normal = g_normal[metrica].dropna()
    v_generoso = g_generoso[metrica].dropna()

    print(f"\n {metrica}")

    stat, p = shapiro(v_exigente)
    print(f"Test de Shapiro-Wilk — exigente — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    stat, p = shapiro(v_normal)
    print(f"Test de Shapiro-Wilk — normal — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    stat, p = shapiro(v_generoso)
    print(f"Test de Shapiro-Wilk — generoso — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")


 MAE
Test de Shapiro-Wilk — exigente — MAE: estadístico=0.960, p-valor=0.000
Test de Shapiro-Wilk — normal — MAE: estadístico=0.928, p-valor=0.000
Test de Shapiro-Wilk — generoso — MAE: estadístico=0.941, p-valor=0.000

 CG@10
Test de Shapiro-Wilk — exigente — CG@10: estadístico=0.599, p-valor=0.000
Test de Shapiro-Wilk — normal — CG@10: estadístico=0.654, p-valor=0.000
Test de Shapiro-Wilk — generoso — CG@10: estadístico=0.483, p-valor=0.000

 DCG@10
Test de Shapiro-Wilk — exigente — DCG@10: estadístico=0.767, p-valor=0.000
Test de Shapiro-Wilk — normal — DCG@10: estadístico=0.761, p-valor=0.000
Test de Shapiro-Wilk — generoso — DCG@10: estadístico=0.559, p-valor=0.000

 NDCG@10
Test de Shapiro-Wilk — exigente — NDCG@10: estadístico=0.377, p-valor=0.000
Test de Shapiro-Wilk — normal — NDCG@10: estadístico=0.519, p-valor=0.000
Test de Shapiro-Wilk — generoso — NDCG@10: estadístico=0.307, p-valor=0.000


c:\Users\esper\Desktop\tp-recomendacion\venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 21331.
  res = hypotest_fun_out(*samples, **kwds)


Ninguna de las métricas evaluadas mediante el test de Shapiro-Wilk presentó un p-valor mayor a 0,05, por lo que se rechaza la hipótesis de normalidad para todas las métricas.

In [24]:
import scipy.stats as stats

for metrica in metricas:
    
    v_exigente = g_exigente[metrica].dropna()
    v_normal = g_normal[metrica].dropna()
    v_generoso = g_generoso[metrica].dropna()
    
    stat, p = stats.levene(v_exigente, v_normal, v_generoso, center='median')
    print(f"Test de Levene para {metrica}: Estadístico={stat:.3f}, p-valor={p:.3f}")

Test de Levene para MAE: Estadístico=708.280, p-valor=0.000
Test de Levene para CG@10: Estadístico=203.907, p-valor=0.000
Test de Levene para DCG@10: Estadístico=236.144, p-valor=0.000
Test de Levene para NDCG@10: Estadístico=142.519, p-valor=0.000


Mediante el test de Levene se determinó que las métricas de los grupos separados según la longitud del historial no presentan homocedasticidad.

Dado que no se cumplen los supuestos de normalidad ni de homocedasticidad, se recurre al test no paramétrico de Kruskal-Wallis

In [27]:
alpha = 0.05

for metrica in metricas:
    # Extracción de valores limpios
    v_exigente = g_exigente[metrica].dropna()
    v_normal = g_normal[metrica].dropna()
    v_generoso = g_generoso[metrica].dropna()
    
    #Test de shapiro-wilk
    stat, p = stats.kruskal(v_exigente, v_normal, v_generoso)
    print(f"\nTest de Kruskal-Wallis — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    if p > alpha:
        print("No hay suficiente evidencia para rechazar la hipótesis nula.")
        print(f"No hay una diferencia significativa en {metrica} entre los grupos de exigencia.")
    else:
        print("Se rechaza la hipótesis nula.")
        print(f"Existe una diferencia significativa en {metrica} entre los grupos de exigencia.")


Test de Kruskal-Wallis — MAE: estadístico=4779.156, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en MAE entre los grupos de exigencia.

Test de Kruskal-Wallis — CG@10: estadístico=3011.530, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en CG@10 entre los grupos de exigencia.

Test de Kruskal-Wallis — DCG@10: estadístico=3599.330, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en DCG@10 entre los grupos de exigencia.

Test de Kruskal-Wallis — NDCG@10: estadístico=463.956, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en NDCG@10 entre los grupos de exigencia.


En cuanto métricas evaluadas, el p-valor es de 0.00. Esto confirma la existencias de diferencias estadisticamente significativas según la exigencia del usuario.

El desempeño del modelo está fuertemente condicionado por la conducta del usuario. El algoritmo sufre una penalización crítica frente a usaurios exigentes, donde se dispara el error de predicción y disminuye drásticamente el volumen de recomendaciones acertadas.

# Conclusión general del SVD

En conclusión, SVD no presenta un comportamiento completamente equitativo entre los distintos grupos de usuarios. Si bien mantiene un desempeño similar en algunos aspectos, existen diferencias en la utilidad de las recomendaciones y en el error de predicción asociadas principalmente al historial de interacciones y al grado de exigencia.

# Impacto En modificaciones simples:


#### Grupo Etario (test = 0.1) 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Adultos | 1.319 | 17.376 | 12.215 | 0.986 |
| Jóvenes | 1.341 | 14.884 | 11.238 | 0.988 |
| Mayores | 1.257 | 13.16 | 10.385 | 0.991 |

#### Historial de Interacciones (test = 0.1)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
|Corto|1.409|7.722|7.633|0.999|
|Largo|1.324|17.778|12.446|0.984|


#### Grupo de Exigencia (test = 0.1)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
|Exigente|2.362|7.696|6.380|0.991|
|Normal|1.172|16.242|11.756|0.986|
|Generoso|1.710|16.898|13.090|0.996|

#### Grupo Etario (test = 0.2) 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Adultos | 1.344 | 19.934 | 13.267 | 0.981 |
| Jóvenes | 1.376 | 17.340 | 12.248 | 0.984 |
| Mayores | 1.327 | 14.940 | 11.256 | 0.988 |

#### Historial de Interacciones (test = 0.2)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
|Corto|1.404|7.817|7.687|0.999|
|Largo|1.345|21.603|14.117|0.977|


#### Grupo de Exigencia (test = 0.2)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
|Exigente|2.492|8.712|6.842|0.985|
|Normal|1.143|18.592|12.753|0.982|
|Generoso|1.827|17.752|13.481|0.995|

#### Grupo Etario (test = 0.3) 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Adultos | 1.365 | 21.428 | 13.871 | 0.978 |
| Jóvenes | 1.382 | 18.729 | 12.842 | 0.983 |
| Mayores | 1.338 | 16.075 | 11.733 | 0.985 |

#### Historial de Interacciones (test = 0.3)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Corto | 1.415 | 7.875 | 7.718 | 0.999 |
| Largo | 1.354 | 24.496 | 15.378 | 0.972 |


#### Grupo de Exigencia (test = 0.3)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Exigente | 2.569 | 9.303 | 7.076 | 0.983 |
| Normal | 1.120 | 20.087 | 13.387 | 0.979 |
| Generoso | 1.905 | 18.157 | 13.674 | 0.995 |

Efectos detectados:
* El error absoluto sube en casi todos los grupos a medida que crece el test set.
    * Al contar con menos interacciones para la fase de ajuste, SVD pierde capacidad para estimar con exactitud los vectores latentes de usarios e ítems, castigando la precisión de las puntuaciones predichas.

* Aumento artificial de la utilidad acumulada (CG y DCG)
    *  Al subir el porcentaje de test, hay más ítems reales disponibles en la bolsa de evaluación de cada usuario, por lo que las recomendaciones tienen mayor probabilidad de sumar coincidencias relevantes.

    * Excepción: En grupos con historial corto, el CG permanece casi plano. Esto muestra que el problema del cold-start frena el rendimiento en estos casos sin importar cuántos datos se reserven para el test.

* El NDCG tiende a disminuir de forma progresiva en la mayoría de los grupos.
    * Al haber más ítems en el conjunto de pruebas, hace que la métrica normalizada sea más estricta.

* Modificar el split no altera las relaciones relativas entre subgrupos.
    * El SVD sigue funcionando mejor en usarios con historial largo que en corto.
    * Los usuarios exigentes siguen sufriendo el MAE mas alto.
    * Los grupos de adultos continuan recibiendo listas de recomendaciones con mayor contenido relevante.  